# 03 · 相空间分析 (postpro 替代)

选择 z 位置 (步进器 ◀◀ ◀ ▶ ▶▶, 对应原 postpro 的步进功能) -> 统计表 +
相空间图自动刷新; 下方合并了原 07 的切片/BFF/导出。发射度单位
"π mm mrad" 与 ASTRA 打印一致。

In [ ]:
%run _bootstrap.py

In [ ]:
from astra_tools.widgets.selectors import discover_sim_runs, PhaseStepper

runs = discover_sim_runs(SIM_DIR)
if not runs:
    raise SystemExit("工作目录没有 ASTRA 输出 — 请先运行 02_astra.ipynb")
stem = sorted(runs)[0]
phase_files = sorted(runs[stem]["001"][t] for t in runs[stem]["001"]
                     if t.lstrip("-").isdigit())
if not phase_files:
    raise SystemExit("没有相空间输出 (OUTPUT 中 PhaseS=F?) — 请先运行 02_astra.ipynb")
print("发现相空间文件: %d 个 (stem=%s)" % (len(phase_files), stem))

# 步进器: 滑块 + ◀◀ ◀ ▶ ▶▶, 对应 postpro 的 z 位置步进功能
stepper = PhaseStepper(phase_files)
stepper

In [ ]:
# 统计表 + 相空间图: 随步进自动刷新
from pathlib import Path
from IPython.display import display
from ipywidgets import interactive_output
import ipywidgets as widgets
from astra_tools.io import read_distribution
from astra_tools.analysis.statistics import compute_statistics
from astra_tools.widgets.panels import distribution_summary_html, stats_table_html, display_bz_warning
from astra_tools.plot.phase_space import plot_transverse_phase_space, plot_phase_space

out = widgets.Output()

def _update(i):
    with out:
        out.clear_output(wait=True)
        dist = read_distribution(stepper.path)
        distribution_summary_html(dist)
        display_bz_warning(SIM_DIR)
        stats_table_html(compute_statistics(dist))
        plot_transverse_phase_space(dist)
        # 高能束流 (1 GeV, εn=1 um) 的 x' 只有 ~1 urad, 原始图必然
        # 是一条贴地横线 (物理正确, 与 Xemit 一致); normalize=True
        # 除以各自 sigma 后结构清晰可见
        plot_phase_space(dist, plane="x", normalize=True,
                         title="x-x' normalized")
        plot_phase_space(dist, plane="z")

display(interactive_output(_update, {"i": stepper.index}))
_update(stepper.index.value)

# 下方单元 (6D/切割/切片/BFF/导出) 使用这个 dist; 步进到新 z 后
# 重跑本单元及需要更新的下方单元即可
dist = read_distribution(stepper.path)
print("当前文件:", stepper.path.name)

**步进器用法**: 点击 ◀ / ▶ (或 ◀◀ / ▶▶, 或直接拖滑块) 切换 z 位置;
上方统计表与三张相空间图**自动刷新**, 下方 6D 全景/切割/切片/BFF/
导出等单元在步进后重跑即可 (与 postpro 的操作逻辑一致: 步进改变
"当前束团", 再选择要看的图)。

In [ ]:
# 6D 全景与投影
from astra_tools.plot.overview import plot_overview, plot_transverse_profile
from astra_tools.plot.distributions import plot_distributions, plot_energy_distribution
fig, _ = plot_overview(dist)
plot_transverse_profile(dist)
plot_distributions(dist)
plot_energy_distribution(dist)

In [ ]:
# z-plot (所有粒子沿束线的位置, 含丢失粒子)
from astra_tools.plot.advanced_plots import plot_z_plot
plot_z_plot(dist)

In [ ]:
# 相空间切割 (postpro 5.6.4): 修改窗口后重跑本单元
from astra_tools.analysis.cuts import cut_distribution
from astra_tools.widgets.panels import stats_table_html
from astra_tools.analysis.statistics import compute_statistics

dist_cut, mask = cut_distribution(dist, x_range=(-1e-3, 1e-3))
print("切割后 (x ±1 mm): 保留 %d/%d 粒子" % (dist_cut.n_active, dist.n_active))
stats_table_html(compute_statistics(dist_cut))

In [ ]:
# 3D slice 椭圆与失配参数
from astra_tools.plot.advanced_plots import plot_slice_ellipses_3d, plot_slice_mismatch
plot_slice_ellipses_3d(dist, n_slices=10)
plot_slice_mismatch(dist, n_slices=10)

## 纵向切片 / BFF / 数据导出 (原 07 高级分析)

In [ ]:
from astra_tools.analysis.slices import compute_slice_analysis
from astra_tools.plot.slice_plots import plot_slice_dashboard
sa = compute_slice_analysis(dist, n_slices=20)
plot_slice_dashboard(sa)

In [ ]:
from astra_tools.analysis.bff import compute_bff
from astra_tools.plot.bff_plots import plot_bff_with_amplitude
bff = compute_bff(dist.filter_active().z, dist.filter_active().charge,
                  kmin=10, kmax=1e5, nk=150, detect_features=True)
plot_bff_with_amplitude(bff)

In [ ]:
# slice 失配参数 (zeta >= 1, 匹配越好越接近 1)
from astra_tools.plot.advanced_plots import plot_slice_mismatch
plot_slice_mismatch(dist, n_slices=20)

In [ ]:
from astra_tools.export import export_distribution, export_statistics
from astra_tools.analysis.statistics import compute_statistics
out = SIM_DIR / "export"
print("导出目录:", out)
print(export_distribution(dist, out))
print(export_statistics(compute_statistics(dist), out))

## 束包络 + 孔径几何叠加 (Aperture 算例)

In [ ]:
from astra_tools.plot.advanced_plots import (
    aperture_elements, plot_envelope_with_aperture)
from astra_tools.namelist.parse import parse_namelists
from astra_tools.io.astra_emit import read_emit_files

ap = parse_namelists(PROJECT_ROOT / "examples/Aperture/astra.in")["APERTURE"]
emit_ap = read_emit_files(str(PROJECT_ROOT / "examples/Aperture/golden/astra"))
plot_envelope_with_aperture(emit_ap, aperture_elements(ap))

## 核心电荷分数曲线 (核心束长/发射度 vs 电荷分数)

In [ ]:
from astra_tools.plot.advanced_plots import plot_core_fraction_curves
plot_core_fraction_curves(dist)